# Python for Data Visualization — An Introduction

Python has become the *lingua franca* of data analysis and visualization. Its strength is not a single tool but a **stack of libraries** that interoperate cleanly: you load and reshape data with one, do the math with another, model it with a third, and draw it with two more — all passing around the same handful of core objects (arrays and DataFrames).

This notebook introduces the five workhorse libraries of the scientific Python stack and shows *why* and *when* you reach for each one:

| Library | Role | Conventional alias |
|---|---|---|
| **NumPy** | Fast numerical arrays & math | `np` |
| **pandas** | Tabular data (load, clean, reshape) | `pd` |
| **Matplotlib** | The base plotting engine | `plt` |
| **seaborn** | High-level statistical charts | `sns` |
| **statsmodels** | Statistical models & inference | `sm` |

### Why this stack?

- **It's the standard.** Tutorials, Stack Overflow answers, and documentation all assume these libraries and these aliases. Following the conventions makes your code instantly readable to other people.
- **The pieces fit together.** A pandas `DataFrame` is built on NumPy arrays; seaborn draws on top of Matplotlib and accepts DataFrames directly; statsmodels accepts both. You rarely have to convert formats manually.
- **From raw numbers to publication figures** in one environment — no jumping between a spreadsheet, a stats package, and a drawing tool.

## Setting up: the standard imports & their conventions

Almost every data project starts with the same block of imports. These aliases (`np`, `pd`, `plt`, `sns`, `sm`) are **community conventions** — they are not required by Python, but using them is expected. When someone sees `pd.read_csv(...)` they instantly know you mean pandas.

> **Note on statsmodels:** the convention is `import statsmodels.api as sm` rather than `import statsmodels as sm`. The public, user-facing functions (regression models, statistical tests, sample datasets) live under the `.api` submodule, so importing the bare package gives you very little to work with. Both lines are shown below for reference; we use the `.api` form throughout.

In [ ]:
# The conventional scientific-Python imports
import numpy as np                 # numerical arrays & math
import pandas as pd                # tabular data
import matplotlib.pyplot as plt    # base plotting
import seaborn as sns              # statistical visualization

# statsmodels: the user asked for `import statsmodels as sm`,
# but the established convention exposes the API submodule:
import statsmodels.api as sm       # statistical models & tests

# A couple of niceties used throughout the notebook
sns.set_theme(style="whitegrid")   # apply seaborn's styling to all matplotlib plots
rng = np.random.default_rng(42)    # reproducible random numbers

print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("seaborn    :", sns.__version__)
print("statsmodels:", sm.version.version)

## 1. NumPy — `np`

**What it is:** NumPy provides the `ndarray`, a fast, memory-efficient N-dimensional array, plus a huge library of vectorized math (trigonometry, linear algebra, statistics, random sampling).

**Why use it:** Pure Python lists are slow for numeric work because every element is a boxed object. NumPy stores data in contiguous memory and runs operations in compiled C, so it's often **10–100× faster** and lets you write math on whole arrays at once (*vectorization*) instead of writing loops. It's also the foundation everything else is built on — pandas, Matplotlib, seaborn and statsmodels all speak NumPy arrays internally.

The example below generates the *x* values and curves we'll plot later — note there is **no `for` loop**; `np.sin(x)` operates on the entire array.

In [ ]:
# Create an evenly spaced array of 100 points between 0 and 2*pi
x = np.linspace(0, 2 * np.pi, 100)

# Vectorized math: these apply to every element at once, no loops needed
sine = np.sin(x)
cosine = np.cos(x)

print("x shape:", x.shape, "| dtype:", x.dtype)
print("first 5 x   :", x[:5].round(3))
print("first 5 sin :", sine[:5].round(3))

# Aggregations are one-liners
print("\nmean of sine :", sine.mean().round(4))
print("max  of cosine:", cosine.max().round(4))

## 2. pandas — `pd`

**What it is:** pandas gives you the `DataFrame` — a labeled, spreadsheet-like table with named columns of (possibly) different types — and the `Series` (a single column). It reads from CSV, Excel, SQL, JSON, Parquet and more.

**Why use it:** Real data is rarely a clean array of numbers; it has column names, missing values, dates, and categories. pandas is built for **loading, cleaning, filtering, grouping, and aggregating** that kind of messy tabular data with concise, readable code. For visualization it matters because seaborn and statsmodels both accept DataFrames directly — you select the columns by name and let the plotting library do the rest.

Below we build a small DataFrame and use `groupby` to summarize it — the kind of step that almost always precedes a chart.

In [ ]:
# Build a small DataFrame from a dictionary of columns
df = pd.DataFrame({
    "city":   ["Austin", "Austin", "Dallas", "Dallas", "Houston", "Houston"],
    "month":  ["Jan", "Feb", "Jan", "Feb", "Jan", "Feb"],
    "sales":  [120, 135, 98, 110, 150, 162],
    "returns": [10, 8, 12, 9, 14, 11],
})

# Quick look at the table
display(df)

# A derived column (vectorized, just like NumPy)
df["net_sales"] = df["sales"] - df["returns"]

# Group + aggregate: total net sales per city
summary = df.groupby("city")["net_sales"].sum().sort_values(ascending=False)
print("\nNet sales by city:")
print(summary)

## 3. Matplotlib — `plt`

**What it is:** Matplotlib is the foundational plotting library for Python. `matplotlib.pyplot` (aliased `plt`) is its MATLAB-style interface for building figures line by line.

**Why use it:** It is the **base layer** almost every other Python plotting tool (including seaborn and pandas' `.plot()`) is built on, and it gives you **total control** — every tick, label, color, and annotation can be customized. The recommended modern style is the *object-oriented* approach: create a `fig` and `ax` with `plt.subplots()`, then call methods on `ax`. Use Matplotlib when you need a custom or finely-tuned figure.

Here we plot the NumPy sine/cosine arrays from section 1.

In [ ]:
# Object-oriented Matplotlib: create a figure and axes, then draw on the axes
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(x, sine,   label="sin(x)", linewidth=2)
ax.plot(x, cosine, label="cos(x)", linewidth=2, linestyle="--")

# Labels, title, legend — the "total control" Matplotlib gives you
ax.set_title("Sine and Cosine (plotted with Matplotlib)")
ax.set_xlabel("x (radians)")
ax.set_ylabel("value")
ax.axhline(0, color="gray", linewidth=0.8)  # reference line at y=0
ax.legend()

plt.tight_layout()
plt.show()

## 4. seaborn — `sns`

**What it is:** seaborn is a high-level statistical visualization library built *on top of* Matplotlib. It turns common, complex plots (distributions, regressions, categorical comparisons, heatmaps) into single function calls.

**Why use it:** Where Matplotlib makes you assemble a chart piece by piece, seaborn is **DataFrame-aware** — you pass a DataFrame plus column names (`x=`, `y=`, `hue=`) and it handles grouping, coloring, legends, and attractive default styling for you. It also computes statistics as part of plotting (e.g. confidence intervals, regression lines, kernel density estimates). Reach for seaborn when you want a good-looking, statistically meaningful chart with minimal code. Because it returns Matplotlib axes, you can still fine-tune with `plt` afterward.

We load seaborn's built-in `tips` dataset and draw two charts: a scatter with a fitted regression line, and a boxplot by category.

In [ ]:
# seaborn ships with sample datasets — handy for demos
tips = sns.load_dataset("tips")
display(tips.head())

# Two plots side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# (a) Scatter + regression line in ONE call, colored by smoker status
sns.regplot(data=tips, x="total_bill", y="tip", ax=axes[0],
            scatter_kws={"alpha": 0.4})
axes[0].set_title("Tip vs. total bill (with regression line)")

# (b) Distribution of tips across days, split by sex — a single call
sns.boxplot(data=tips, x="day", y="tip", hue="sex", ax=axes[1])
axes[1].set_title("Tip distribution by day")

plt.tight_layout()
plt.show()

## 5. statsmodels — `sm`

**What it is:** statsmodels is a library for **statistical modeling and inference**: linear and generalized linear regression, time-series models (ARIMA), hypothesis tests, and detailed statistical summaries.

**Why use it:** seaborn can *draw* a regression line, but it won't tell you the slope, its standard error, the R², or whether the relationship is statistically significant. statsmodels fills that gap — it's the tool when you care about the **numbers behind the picture**: coefficients, p-values, confidence intervals, and goodness-of-fit. It's the closest thing in Python to the output of R or a dedicated stats package, which makes it the natural companion to a seaborn chart.

Below we fit an Ordinary Least Squares (OLS) regression of `tip` on `total_bill` and print the full summary. Note the `sm.add_constant` step — statsmodels does **not** add an intercept term automatically.

In [ ]:
# Fit an Ordinary Least Squares regression: tip ~ total_bill
y = tips["tip"]                          # response / dependent variable
X = tips["total_bill"]                   # predictor / independent variable
X = sm.add_constant(X)                   # add the intercept column (required!)

model = sm.OLS(y, X).fit()               # build and fit the model

# The full statistical report: coefficients, p-values, R-squared, etc.
print(model.summary())

# Pull out individual numbers programmatically
print("\nSlope (tip per $1 of bill):", round(model.params['total_bill'], 4))
print("R-squared                 :", round(model.rsquared, 4))

## Putting it together — the typical workflow

These five libraries aren't competitors; they form a **pipeline**, each handing off to the next:

```
NumPy        →  pandas        →  seaborn / Matplotlib  →  statsmodels
(raw numbers)   (clean table)    (explore visually)       (quantify & test)
```

1. **NumPy** holds the raw numeric arrays and does the fast math.
2. **pandas** loads, cleans, and reshapes that data into labeled tables.
3. **seaborn** (with **Matplotlib** underneath) lets you *see* patterns quickly, and Matplotlib polishes the figure for publication.
4. **statsmodels** *quantifies* those patterns with rigorous statistics.

### When to reach for which

| Need | Use |
|---|---|
| Fast numeric arrays / math | **NumPy** |
| Load, clean, group tabular data | **pandas** |
| Full control over a custom figure | **Matplotlib** |
| A quick, attractive statistical chart | **seaborn** |
| Coefficients, p-values, model fit | **statsmodels** |

**Next steps:** try swapping in your own CSV with `pd.read_csv("yourfile.csv")`, then explore it with a seaborn plot and confirm what you see with a statsmodels model.

## Bonus: IPython magic commands

Jupyter notebooks run on **IPython**, an enhanced Python interpreter. On top of normal Python, IPython adds *magic commands* — special commands prefixed with `%` that control the environment, time code, debug errors, and more. They aren't Python syntax; they're convenience tools provided by IPython.

There are two flavors:

- **Line magics** start with a single `%` and operate on the rest of the line — e.g. `%timeit sum(range(100))`.
- **Cell magics** start with `%%` and operate on the *entire cell* below them — e.g. `%%timeit` times every line in the cell.

> Tip: append `?` to any magic (e.g. `%timeit?`) to see its documentation, and run `%lsmagic` to list every available magic.

### Table 2-2. Some frequently used IPython magic commands

| Command | Description |
|---|---|
| `%quickref` | Display the IPython Quick Reference Card |
| `%magic` | Display detailed documentation for all of the available magic commands |
| `%debug` | Enter the interactive debugger at the bottom of the last exception traceback |
| `%hist` | Print command input (and optionally output) history |
| `%pdb` | Automatically enter debugger after any exception |
| `%paste` | Execute preformatted Python code from clipboard |
| `%cpaste` | Open a special prompt for manually pasting Python code to be executed |
| `%reset` | Delete all variables/names defined in interactive namespace |
| `%page OBJECT` | Pretty-print the object and display it through a pager |
| `%run script.py` | Run a Python script inside IPython |
| `%prun statement` | Execute `statement` with cProfile and report the profiler output |
| `%time statement` | Report the execution time of a single statement |
| `%timeit statement` | Run a statement multiple times to compute an ensemble average execution time; useful for timing code with very short execution time |
| `%who`, `%who_ls`, `%whos` | Display variables defined in interactive namespace, with varying levels of information |

### Why they're useful here

When you're building visualizations, a few of these come up constantly:

- **`%timeit`** — compare whether a NumPy vectorized version is really faster than a Python loop.
- **`%who` / `%whos`** — see which DataFrames and arrays you've already defined without scrolling back up.
- **`%run`** — pull a data-prep script into the notebook session.
- **`%debug` / `%pdb`** — drop into the debugger when a plotting or modeling call raises an exception.
- **`%matplotlib inline`** — the classic magic that renders Matplotlib figures directly in the notebook (modern Jupyter does this by default, but you'll still see it in many tutorials).

In [ ]:
# A few safe magics you can run right now.
# (Magics like %debug, %paste or %reset are interactive/destructive, so we skip them here.)

# %timeit shows why NumPy vectorization matters: time a pure-Python loop...
%timeit [i ** 2 for i in range(1000)]

# ...versus the equivalent NumPy operation
arr = np.arange(1000)
%timeit arr ** 2

# %who lists the names currently defined in this session
%who

# List every available magic command
%lsmagic